# SENSERO dataset split — MGRS-tile stratified

`dataset_split.ipynb`: the multi-label stratified split is run **per MGRS tile**, then concatenated. The rare-class enforcement and exact-target balancing run on the concatenated result.

## Why

Random multi-label stratification ignores geography. Patches from the same MGRS tile (and therefore the same neighbourhood) can end up in train *and* test, which makes the test set easier than it should be. 

Splitting per tile means:

- Every MGRS tile contributes patches to train, val, **and** test in the configured ratio
- Train and test patches can still come from the same tile (so this is **not** full spatial separation), but the test set is spread across the country instead of clustered
- No new dependencies — MGRS tile is already in `BaseFilename` (the `_T34TFS_` segment)

## What did NOT change

- Rare-class rules (k=1, k=2, k=3 enforcement)
- Exact target counts (7000 / 1500 / 1500)
- Propagation across patch sizes
- Random seed (41) — reproducibility preserved within each tile's split


In [13]:
# ============================================================
# CELL 1: Configuration and Imports
# ============================================================

import os, re, json, sys, time, shutil, tempfile
import numpy as np
import pandas as pd
import rasterio
import concurrent.futures
from tqdm import tqdm
from collections import defaultdict
from datetime import datetime
from typing import Dict, List, Tuple, Set

# Hard requirement - no silent fallback
try:
    from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
except ImportError:
    raise ImportError(
        "iterative-stratification is required for reproducible multi-label splits. "
        "Install with: pip install iterative-stratification"
    )

# ---------- CONFIG ----------
ROOT = "/home/ubuntu/SENSERO/GeoTiff/"

CANONICAL_SIZE = 128
ALL_SIZES = [64, 112, 120, 128, 224, 256, 280, 336]

MIN_CLASS_PCT = 2.0
SPLIT_RATIOS = (0.7, 0.15, 0.15)
RANDOM_STATE = 42
RNG = np.random.default_rng(RANDOM_STATE)

TARGET_TRAIN = 7000
TARGET_VAL   = 1500
TARGET_TEST  = 1500

# NEW: MGRS tile extraction regex.
# Filenames look like: S2A_MSIL2A_20181014T093031_N0500_R136_T34TFS_10056_3577
# We want "34TFS" — the part right after "_T" before the next "_".
MGRS_RE = re.compile(r"_T([0-9]{1,2}[A-Z]{3})_")

# Minimum patches a tile must have to be split internally. Tiles with fewer
# patches all go to TRAIN (too few for meaningful 70/15/15 stratification).
MIN_TILE_FOR_SPLIT = 10

CHECKPOINT_FILE = os.path.join(ROOT, "canonical_checkpoint.parquet")
SWAP_LOG_FILE   = os.path.join(ROOT, "swap_log.csv")
RARE_FAILURES_FILE = os.path.join(ROOT, "rare_class_failures.csv")

MAX_REPAIR_ITER = 10

# Tracking
rare_class_failures = []

# Clear old swap log at start of run (Jupyter re-runs are common)
if os.path.exists(SWAP_LOG_FILE):
    os.remove(SWAP_LOG_FILE)

print(f"Config loaded. Root: {ROOT}")
print(f"Canonical size: {CANONICAL_SIZE}px, sizes: {ALL_SIZES}")
print(f"Split ratios: {SPLIT_RATIOS}, targets: {TARGET_TRAIN}/{TARGET_VAL}/{TARGET_TEST}")
print(f"MGRS tile extraction: regex {MGRS_RE.pattern!r}")
print(f"Min patches per tile to internally split: {MIN_TILE_FOR_SPLIT}")


Config loaded. Root: /home/ubuntu/SENSERO/GeoTiff/
Canonical size: 128px, sizes: [64, 112, 120, 128, 224, 256, 280, 336]
Split ratios: (0.7, 0.15, 0.15), targets: 7000/1500/1500
MGRS tile extraction: regex '_T([0-9]{1,2}[A-Z]{3})_'
Min patches per tile to internally split: 10


In [14]:
# ============================================================
# CELL 2: Save & Logging Utilities
# ============================================================

def atomic_save_parquet(df, path):
    tmp = path + ".tmp"
    df.to_parquet(tmp)
    os.replace(tmp, path)

def atomic_save_csv(df, path):
    tmp = path + ".tmp"
    df.to_csv(tmp, index=False)
    os.replace(tmp, path)

def log_swap(action, idx_from, src, dst, df):
    """Log a direct or swap move to CSV."""
    ts = datetime.now().isoformat()
    fn = df.at[idx_from, "BaseFilename"]
    entry = pd.DataFrame([{
        "timestamp": ts,
        "action": action,
        "index": idx_from,
        "BaseFilename": fn,
        "from_split": src,
        "to_split": dst
    }])

    if not os.path.exists(SWAP_LOG_FILE):
        entry.to_csv(SWAP_LOG_FILE, index=False)
    else:
        entry.to_csv(SWAP_LOG_FILE, mode="a", header=False, index=False)

print("Utilities defined.")


Utilities defined.


In [15]:
# ============================================================
# CELL 3: TIFF Loading (Parallel)
# ============================================================

def find_clc_dir(size):
    d = os.path.join(ROOT, f"Patch_{size}", "CLC_reference")
    if not os.path.exists(d):
        raise FileNotFoundError(f"[MISSING] {d}")
    return d

def strip_clc(name):
    base = os.path.splitext(name)[0]
    return re.sub(r"_CLC$", "", base, flags=re.IGNORECASE)

def extract_single_tif(tif_path):
    try:
        with rasterio.open(tif_path) as src:
            arr = src.read(1)
        arr = arr.astype(np.int32)
        mask = arr > 0
        if not np.any(mask):
            return None
        vals, cnts = np.unique(arr[mask], return_counts=True)
        pct = (cnts.astype(float) / cnts.sum()) * 100
        classes = [int(v) for v, p in zip(vals, pct) if p >= MIN_CLASS_PCT]
        if not classes:
            return None
        base = strip_clc(os.path.basename(tif_path))
        return base, sorted(classes)
    except Exception:
        return None

def collect_labels(size):
    print(f"[INFO] Loading TIFFs for {size}px ...")
    root = find_clc_dir(size)
    tifs = []
    for scene in os.listdir(root):
        sd = os.path.join(root, scene)
        if not os.path.isdir(sd):
            continue
        for f in os.listdir(sd):
            if f.lower().endswith(".tif"):
                tifs.append(os.path.join(sd, f))

    rows = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=12) as exe:
        futures = {exe.submit(extract_single_tif, p): p for p in tifs}
        for fut in tqdm(concurrent.futures.as_completed(futures),
                        total=len(futures),
                        desc=f"Extracting classes @ {size}px"):
            res = fut.result()
            if res is None:
                continue
            base, classes = res
            rows.append({"BaseFilename": base, "codes_list": classes})

    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError(f"No valid TIFFs at {size}px")

    df = df.groupby("BaseFilename", as_index=False)["codes_list"].agg(
        lambda lsts: sorted(set(sum(lsts, [])))
    )

    print(f"[INFO] {size}px -> {len(df)} patches loaded.")
    return df

print("TIFF loading functions defined.")


TIFF loading functions defined.


In [16]:
# ============================================================
# CELL 4: Multilabel Matrix & MGRS-Stratified Split
# ============================================================

def build_multilabel_matrix(df):
    all_codes = sorted({c for lst in df["codes_list"] for c in lst})
    idx = {c: i for i, c in enumerate(all_codes)}
    Y = np.zeros((len(df), len(all_codes)), dtype=np.int8)
    for i, codes in enumerate(df["codes_list"]):
        for c in codes:
            Y[i, idx[c]] = 1
    return Y, all_codes

def extract_mgrs(name):
    """Pull the MGRS tile id (e.g. '34TFS') from a SENSERO BaseFilename."""
    m = MGRS_RE.search(name)
    if m is None:
        raise ValueError(f"Could not extract MGRS tile from filename: {name!r}")
    return m.group(1)

def _split_one_block(Y_block, ratios):
    """Run the original two-step iterstrat split on a single block of rows.

    Returns three index arrays (positions within Y_block) for train/val/test.
    """
    n = len(Y_block)
    # Edge case: stratifier needs both classes present; very small blocks
    # can fail. Caller handles MIN_TILE_FOR_SPLIT, but be defensive.
    m1 = MultilabelStratifiedShuffleSplit(
        n_splits=1, test_size=(1 - ratios[0]), random_state=RANDOM_STATE)
    tr, tmp = next(m1.split(np.zeros((n, 1)), Y_block))

    rel_test = ratios[2] / (ratios[1] + ratios[2])
    m2 = MultilabelStratifiedShuffleSplit(
        n_splits=1, test_size=rel_test, random_state=RANDOM_STATE)
    vr, te = next(m2.split(np.zeros((len(tmp), 1)), Y_block[tmp]))

    return tr, tmp[vr], tmp[te]

def stratified_multilabel_split_by_mgrs(df, Y, ratios=SPLIT_RATIOS):
    """Per-MGRS-tile multi-label stratified split.

    For each MGRS tile with >= MIN_TILE_FOR_SPLIT patches: split that tile's
    patches 70/15/15 (or whatever ratios) using iterative multi-label
    stratification. Tiles below the threshold contribute all patches to train
    (small tiles can't be split meaningfully and shouldn't appear in test/val).

    Returns three index arrays into the FULL df (not per-tile).
    """
    tiles = df["mgrs"].values
    unique_tiles = sorted(set(tiles))
    print(f"[SPLIT] {len(unique_tiles)} unique MGRS tiles")

    train_idx, val_idx, test_idx = [], [], []
    small_tile_patches = 0
    per_tile_stats = []

    for tile in unique_tiles:
        mask = (tiles == tile)
        block_positions = np.where(mask)[0]  # positions in full df
        Y_block = Y[block_positions]
        n_block = len(block_positions)

        if n_block < MIN_TILE_FOR_SPLIT:
            # Too small to split — all to train
            train_idx.extend(block_positions.tolist())
            small_tile_patches += n_block
            per_tile_stats.append({
                "mgrs": tile, "n": n_block,
                "train": n_block, "val": 0, "test": 0, "note": "small->train"
            })
            continue

        try:
            tr, va, te = _split_one_block(Y_block, ratios)
            train_idx.extend(block_positions[tr].tolist())
            val_idx.extend(block_positions[va].tolist())
            test_idx.extend(block_positions[te].tolist())
            per_tile_stats.append({
                "mgrs": tile, "n": n_block,
                "train": len(tr), "val": len(va), "test": len(te), "note": ""
            })
        except Exception as e:
            # Fall back to all-train if stratification fails for this tile
            print(f"[SPLIT][WARN] Tile {tile} (n={n_block}) stratification failed: {e}. "
                  "Assigning all to train.")
            train_idx.extend(block_positions.tolist())
            per_tile_stats.append({
                "mgrs": tile, "n": n_block,
                "train": n_block, "val": 0, "test": 0, "note": f"fail->train ({e})"
            })

    stats_df = pd.DataFrame(per_tile_stats)
    print(f"\n[SPLIT] Per-tile distribution:")
    print(stats_df.to_string(index=False))
    print(f"\n[SPLIT] {small_tile_patches} patches from small tiles "
          f"(< {MIN_TILE_FOR_SPLIT}) routed to train.")
    print(f"[SPLIT] Totals: train={len(train_idx)}, val={len(val_idx)}, test={len(test_idx)}")

    return np.array(train_idx), np.array(val_idx), np.array(test_idx), stats_df

print("Stratification functions defined.")


Stratification functions defined.


In [17]:
# ============================================================
# CELL 5: Swap Logic & Rare-Class Rules (UNCHANGED)
# ============================================================

def is_abundant(codes):
    return any(200 <= c <= 399 for c in codes)

def choose_swap(df, dst_split, exclude=None, protected_idxs=None):
    cand = df.index[df["split"] == dst_split].tolist()
    if exclude is not None and exclude in cand:
        cand.remove(exclude)
    if protected_idxs:
        cand = [i for i in cand if i not in protected_idxs]
    if not cand:
        return -1
    abundant = [i for i in cand if is_abundant(df.at[i, "codes_list"])]
    pool = abundant if abundant else cand
    return int(RNG.choice(pool))

def move_with_swap(df, idx, src, dst, protected_idxs=None, reason=""):
    if df.at[idx, "split"] != src:
        return False
    df.at[idx, "split"] = dst
    sw = choose_swap(df, dst, exclude=idx, protected_idxs=protected_idxs)
    if sw == -1:
        df.at[idx, "split"] = src
        rare_class_failures.append({
            "patch_idx": int(idx),
            "BaseFilename": df.at[idx, "BaseFilename"],
            "from_split": src,
            "to_split": dst,
            "reason": reason,
            "failure": "no swap candidate available"
        })
        return False
    df.at[sw, "split"] = src
    log_swap("swap", idx, src, dst, df)
    return True

def enforce_rare_classes(df):
    print("[RARE] Applying rare class rules...")
    class_map = defaultdict(list)
    for i, codes in df["codes_list"].items():
        for c in codes:
            class_map[c].append(i)

    protected = set()

    for c, idxs in tqdm(class_map.items(), desc="Rare classes"):
        k = len(idxs)
        if k == 1:
            i = idxs[0]
            if df.at[i, "split"] != "train":
                if move_with_swap(df, i, df.at[i, "split"], "train",
                                  protected_idxs=protected,
                                  reason=f"class {c} k=1 -> train"):
                    protected.add(i)
            else:
                protected.add(i)
        elif k == 2:
            current = [df.at[i, "split"] for i in idxs]
            if "train" not in current:
                if move_with_swap(df, idxs[0], df.at[idxs[0], "split"], "train",
                                  protected_idxs=protected,
                                  reason=f"class {c} k=2 -> ensure train"):
                    protected.add(idxs[0])
            current = [df.at[i, "split"] for i in idxs]
            if "test" not in current:
                for i in idxs:
                    if df.at[i, "split"] != "train":
                        if move_with_swap(df, i, df.at[i, "split"], "test",
                                          protected_idxs=protected,
                                          reason=f"class {c} k=2 -> ensure test"):
                            protected.add(i)
                        break
            for i in idxs:
                if df.at[i, "split"] == "train":
                    protected.add(i)
        elif k == 3:
            for target in ["train", "val", "test"]:
                current = [df.at[i, "split"] for i in idxs]
                if target in current:
                    continue
                moved = False
                for i in idxs:
                    s = df.at[i, "split"]
                    if s != target and current.count(s) >= 2 and i not in protected:
                        if move_with_swap(df, i, s, target,
                                          protected_idxs=protected,
                                          reason=f"class {c} k=3 -> ensure {target}"):
                            protected.add(i)
                            moved = True
                            break
                if not moved:
                    for i in idxs:
                        if df.at[i, "split"] != target and i not in protected:
                            if move_with_swap(df, i, df.at[i, "split"], target,
                                              protected_idxs=protected,
                                              reason=f"class {c} k=3 -> ensure {target} (fallback)"):
                                protected.add(i)
                                break
            for i in idxs:
                protected.add(i)

    print(f"[RARE] {len(protected)} patches protected from rebalancing.")
    if rare_class_failures:
        print(f"[WARN] {len(rare_class_failures)} rare-class assignments failed.")
        pd.DataFrame(rare_class_failures).to_csv(RARE_FAILURES_FILE, index=False)
        print(f"[SAVE] Failures logged to {RARE_FAILURES_FILE}")

    return protected

print("Rare-class enforcement defined.")


Rare-class enforcement defined.


In [18]:
# ============================================================
# CELL 6: Exact Balancing (UNCHANGED)
# ============================================================

def enforce_exact(df, protected=None, train=TARGET_TRAIN, val=TARGET_VAL, test=TARGET_TEST):
    print("[BALANCE] Enforcing exact targets using direct moves...")
    if protected is None:
        protected = set()

    target = {"train": train, "val": val, "test": test}
    iteration = 0
    max_iter = 50000

    while iteration < max_iter:
        iteration += 1
        counts = df["split"].value_counts().to_dict()
        for k in target:
            counts.setdefault(k, 0)

        if iteration % 100 == 1 or counts == target:
            print(f"[BALANCE] Iter {iteration}: train={counts['train']}, "
                  f"val={counts['val']}, test={counts['test']} "
                  f"(target {train}/{val}/{test})")

        if counts == target:
            print("[BALANCE] Targets achieved exactly.")
            return df

        over  = max(counts, key=lambda k: counts[k] - target[k])
        under = min(counts, key=lambda k: counts[k] - target[k])

        if counts[over] - target[over] <= 0:
            print("[BALANCE] No further moves possible (no over-represented split).")
            return df

        candidates = df.index[
            (df["split"] == over) & (~df.index.isin(protected))
        ].tolist()

        if not candidates:
            print(f"[BALANCE] No unprotected patches in {over}; cannot continue. "
                  f"Final counts: {counts}")
            return df

        idx = int(RNG.choice(candidates))
        df.at[idx, "split"] = under
        log_swap("direct_move", idx, over, under, df)

    print(f"[BALANCE] Reached max iterations ({max_iter}). Final: {counts}")
    return df

print("Balancing function defined.")


Balancing function defined.


In [19]:
# ============================================================
# CELL 7: Propagation & Canonical Repair (UNCHANGED)
# ============================================================

def propagate_with_validation(size, df_canon):
    df_s = collect_labels(size)
    names_c = set(df_canon["BaseFilename"])
    names_s = set(df_s["BaseFilename"])
    missing = names_c - names_s
    if missing:
        return False, missing, df_s
    merged = df_s.merge(df_canon[["BaseFilename", "split"]], on="BaseFilename", how="left")
    return True, None, merged

def repair_canonical_missing(df_canon, missing_set, df_s, protected):
    print(f"[REPAIR] Missing: {len(missing_set)} patches -> removing & replacing")
    protected_filenames = {df_canon.at[i, "BaseFilename"] for i in protected if i in df_canon.index}
    truly_missing_protected = missing_set & protected_filenames
    if truly_missing_protected:
        print(f"[REPAIR][WARN] {len(truly_missing_protected)} protected (rare-class) patches "
              f"are missing at other scales. These rare classes may lose coverage.")
    df_canon = df_canon[~df_canon["BaseFilename"].isin(missing_set)].copy().reset_index(drop=True)
    available = list(set(df_s["BaseFilename"]) - set(df_canon["BaseFilename"]))
    if len(available) < len(missing_set):
        raise RuntimeError("Not enough replacement candidates!")
    replacements = RNG.choice(available, size=len(missing_set), replace=False)
    new_rows = []
    for name in replacements:
        row = df_s[df_s["BaseFilename"] == name].iloc[0]
        new_rows.append({
            "BaseFilename": name,
            "codes_list": row["codes_list"],
            "split": "train"
        })
    df_canon = pd.concat([df_canon, pd.DataFrame(new_rows)], ignore_index=True)
    df_canon = enforce_exact(df_canon, protected=set())
    return df_canon

def propagate_all(df_canon, protected):
    print("[PROP] Validating canonical splits across all scales...")
    for attempt in range(MAX_REPAIR_ITER):
        restart = False
        for size in ALL_SIZES:
            if size == CANONICAL_SIZE:
                continue
            ok, missing, df_s = propagate_with_validation(size, df_canon)
            if not ok:
                print(f"[PROP] Scale {size}px: {len(missing)} missing -> repairing canonical.")
                df_canon = repair_canonical_missing(df_canon, missing, df_s, protected)
                restart = True
                break
        if not restart:
            print("[PROP] Canonical valid for all scales.")
            return df_canon
    raise RuntimeError("Too many canonical repair iterations (possible infinite loop).")

print("Propagation functions defined.")


Propagation functions defined.


In [20]:
# ============================================================
# CELL 8: Reporting & Saving (UNCHANGED + per-tile distribution report)
# ============================================================

def rare_class_report(df):
    print("\n====== RARE CLASS DISTRIBUTION ======")
    class_map = defaultdict(list)
    for i, codes in df["codes_list"].items():
        for c in codes:
            class_map[c].append(df.at[i, "split"])
    stats = []
    for c, splits in class_map.items():
        stats.append({
            "class": c, "freq": len(splits),
            "train": splits.count("train"),
            "val": splits.count("val"),
            "test": splits.count("test")
        })
    df_stats = pd.DataFrame(stats).sort_values("freq")
    print(df_stats.to_string(index=False))
    rare = df_stats[df_stats["freq"] <= 3]
    if not rare.empty:
        print("\n--- Rare classes (freq <= 3) ---")
        for _, r in rare.iterrows():
            k = r["freq"]
            ok = True
            if k == 1 and r["train"] < 1: ok = False
            elif k == 2 and (r["train"] < 1 or r["test"] < 1): ok = False
            elif k == 3 and (r["train"] < 1 or r["val"] < 1 or r["test"] < 1): ok = False
            status = "OK" if ok else "FAIL"
            print(f"  class {r['class']}: k={k}, "
                  f"train={r['train']}, val={r['val']}, test={r['test']}  [{status}]")
    print("====================================\n")
    return df_stats

def per_tile_distribution_report(df):
    """NEW: show final per-tile train/val/test counts after all balancing."""
    print("\n====== FINAL PER-MGRS-TILE DISTRIBUTION ======")
    grp = df.groupby(["mgrs", "split"]).size().unstack(fill_value=0)
    for c in ["train", "val", "test"]:
        if c not in grp.columns:
            grp[c] = 0
    grp = grp[["train", "val", "test"]]
    grp["total"] = grp.sum(axis=1)
    print(grp.to_string())

    # Flag tiles that ended up with zero test (or val) — useful sanity check
    no_test = grp[grp["test"] == 0]
    no_val  = grp[grp["val"] == 0]
    if len(no_test):
        print(f"\n[NOTE] {len(no_test)} tiles have zero test patches "
              f"(small tiles or post-balancing artefact).")
    if len(no_val):
        print(f"[NOTE] {len(no_val)} tiles have zero val patches.")
    print("==============================================\n")
    return grp

def save_split(size, df):
    outdir = os.path.join(ROOT, f"Patch_{size}")
    os.makedirs(outdir, exist_ok=True)
    df2 = df.copy()
    df2["codes_list"] = df2["codes_list"].apply(lambda x: "[" + ", ".join(map(str, x)) + "]")
    atomic_save_csv(df2, os.path.join(outdir, "split_summary.csv"))
    atomic_save_parquet(df2, os.path.join(outdir, "split_summary.parquet"))
    print(f"[SAVE] {size}px split saved ({len(df)} patches).")

print("Reporting and saving functions defined.")


Reporting and saving functions defined.


In [21]:
# ============================================================
# CELL 9: Build Canonical Split (MGRS-stratified)
# ============================================================

np.random.seed(RANDOM_STATE)

# 1) Load canonical
df_canon = collect_labels(CANONICAL_SIZE)
Y, all_codes = build_multilabel_matrix(df_canon)
print(f"[INFO] Canonical matrix: {Y.shape[0]} patches x {Y.shape[1]} classes")

# 2) Extract MGRS tile per patch
df_canon["mgrs"] = df_canon["BaseFilename"].apply(extract_mgrs)
print(f"[INFO] Tile counts:")
print(df_canon["mgrs"].value_counts().to_string())

# 3) MGRS-stratified multi-label split
tr, va, te, tile_stats = stratified_multilabel_split_by_mgrs(df_canon, Y)
df_canon["split"] = None
df_canon.loc[tr, "split"] = "train"
df_canon.loc[va, "split"] = "val"
df_canon.loc[te, "split"] = "test"

# Save per-tile split breakdown for the paper / supplementary
tile_stats.to_csv(os.path.join(ROOT, "mgrs_tile_split_stats.csv"), index=False)
print(f"\n[SAVE] Per-tile split stats: {os.path.join(ROOT, 'mgrs_tile_split_stats.csv')}")

print(f"\n[INITIAL] split counts after MGRS-stratified split:")
print(df_canon["split"].value_counts())


[INFO] Loading TIFFs for 128px ...


Extracting classes @ 128px: 100%|███████████████████████████████████████████████| 10000/10000 [00:19<00:00, 525.57it/s]


[INFO] 128px -> 10000 patches loaded.
[INFO] Canonical matrix: 10000 patches x 34 classes
[INFO] Tile counts:
mgrs
34TGS    468
34TFT    463
35TNK    431
34TGQ    429
35TMN    428
34TER    422
34TGT    418
35TMK    413
35TPK    412
35TMM    405
35TNL    404
34TFQ    403
35TML    369
34TGR    367
34TFR    367
34TFS    357
35TLK    351
35TNM    325
34TES    307
35TLJ    267
34TGP    256
35TLN    233
35TLL    216
34TEQ    187
35TNN    168
35TLM    159
34TET    146
35UMP    146
35TPJ     93
35TPL     93
34TDR     92
35TMJ     81
34TFP     76
34UFU     67
34UGU     65
34TDS     49
35TNJ     43
35ULP     24
[SPLIT] 38 unique MGRS tiles

[SPLIT] Per-tile distribution:
 mgrs   n  train  val  test note
34TDR  92     63   14    15     
34TDS  49     34    7     8     
34TEQ 187    132   26    29     
34TER 422    295   60    67     
34TES 307    213   47    47     
34TET 146     98   26    22     
34TFP  76     54   12    10     
34TFQ 403    285   60    58     
34TFR 367    262   52    53     


In [22]:
# ============================================================
# CELL 10: Apply Rare-Class Rules and Exact Balancing
# ============================================================
# NOTE: These run on the already MGRS-stratified result. They may move some
# patches across tile boundaries to satisfy rare-class coverage or exact
# target counts. This is expected and unavoidable — without it, you cannot
# guarantee both rare-class coverage AND exact 7000/1500/1500.

protected = enforce_rare_classes(df_canon)
print(f"\n[AFTER RARE] split counts:")
print(df_canon["split"].value_counts())

df_canon = enforce_exact(df_canon, protected=protected)
print(f"\n[AFTER BALANCE] split counts:")
print(df_canon["split"].value_counts())

atomic_save_parquet(df_canon, CHECKPOINT_FILE)
print(f"[SAVE] Checkpoint: {CHECKPOINT_FILE}")


[RARE] Applying rare class rules...


Rare classes: 100%|████████████████████████████████████████████████████████████████| 34/34 [00:00<00:00, 267053.06it/s]

[RARE] 0 patches protected from rebalancing.

[AFTER RARE] split counts:
split
train    6972
test     1526
val      1502
Name: count, dtype: int64
[BALANCE] Enforcing exact targets using direct moves...
[BALANCE] Iter 1: train=6972, val=1502, test=1526 (target 7000/1500/1500)
[BALANCE] Iter 29: train=7000, val=1500, test=1500 (target 7000/1500/1500)
[BALANCE] Targets achieved exactly.

[AFTER BALANCE] split counts:
split
train    7000
val      1500
test     1500
Name: count, dtype: int64
[SAVE] Checkpoint: /home/ubuntu/SENSERO/GeoTiff/canonical_checkpoint.parquet


In [23]:
# ============================================================
# CELL 11: Propagate & Save All Scales
# ============================================================

df_canon = propagate_all(df_canon, protected)

df_stats = rare_class_report(df_canon)
tile_final = per_tile_distribution_report(df_canon)
tile_final.to_csv(os.path.join(ROOT, "mgrs_tile_split_final.csv"))
print(f"[SAVE] Final per-tile distribution: {os.path.join(ROOT, 'mgrs_tile_split_final.csv')}")

master = df_canon[["BaseFilename", "split"]].copy()
atomic_save_csv(master, os.path.join(ROOT, "master_split_from_tiff_corrected.csv"))
atomic_save_parquet(master, os.path.join(ROOT, "master_split_from_tiff_corrected.parquet"))
print(f"[SAVE] Master split saved.")

print("\n[SAVE] Saving splits for all patch sizes...\n")
for size in ALL_SIZES:
    df_s = collect_labels(size)
    merged = df_s.merge(master, on="BaseFilename", how="left")
    n_unmatched = merged["split"].isna().sum()
    if n_unmatched > 0:
        print(f"[WARN] {size}px: {n_unmatched} patches not in master, assigning to train")
    merged["split"] = merged["split"].fillna("train")
    save_split(size, merged)

print("\nDONE. All scales aligned, exact targets achieved, rare classes preserved,")
print("MGRS tile stratification applied at canonical level.")
print(f"\nFinal canonical split counts:")
print(df_canon["split"].value_counts())


[PROP] Validating canonical splits across all scales...
[INFO] Loading TIFFs for 64px ...


Extracting classes @ 64px: 100%|████████████████████████████████████████████████| 10000/10000 [00:18<00:00, 552.51it/s]


[INFO] 64px -> 10000 patches loaded.
[INFO] Loading TIFFs for 112px ...


Extracting classes @ 112px: 100%|███████████████████████████████████████████████| 10000/10000 [00:17<00:00, 576.50it/s]


[INFO] 112px -> 10000 patches loaded.
[INFO] Loading TIFFs for 120px ...


...

[INFO] 256px -> 10000 patches loaded.
[SAVE] 256px split saved (10000 patches).
[INFO] Loading TIFFs for 280px ...


Extracting classes @ 280px: 100%|███████████████████████████████████████████████| 10000/10000 [00:21<00:00, 475.92it/s]


[INFO] 280px -> 10000 patches loaded.
[SAVE] 280px split saved (10000 patches).
[INFO] Loading TIFFs for 336px ...


Extracting classes @ 336px: 100%|███████████████████████████████████████████████| 10000/10000 [00:21<00:00, 469.38it/s]


[INFO] 336px -> 10000 patches loaded.
[SAVE] 336px split saved (10000 patches).

DONE. All scales aligned, exact targets achieved, rare classes preserved,
MGRS tile stratification applied at canonical level.

Final canonical split counts:
split
train    7000
val      1500
test     1500
Name: count, dtype: int64
